# 11-685 diffusion baseline: Kaggle training and evaluation

This notebook restores an unchanged snapshot of the submitted repository, downloads the course ImageNet-100 archive, trains the existing pixel-space DDPM, and evaluates **one checkpoint** using DDPM and DDIM sampling. It records FID and Inception Score (IS) for 5,000 generated images per sampler. No repository source file or TODO comment is edited here.

**Before running:** choose a Kaggle GPU and enable Internet. Run the source-restore cell first, then the dependency cell, then continue in order. The default is a 1-epoch, 1,000-image pilot; change `RUN_MODE` to `full` for the planned 3-epoch ImageNet-100 run. Save a Kaggle version and download the checkpoint, logs, and metric files before ending the session. This notebook has been prepared but its updated cells have **not** been executed.

The earlier interactive pilot completed at 64×64, batch size 2, 500 updates in about 1.07 minutes, and saved `checkpoint_epoch_0.pth`. That timing is evidence for the pilot only; it does not establish image quality or FID/IS.


In [ ]:
import os, torch
from pathlib import Path
assert torch.cuda.is_available(), "Enable a GPU in Kaggle Settings > Accelerator"
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Working storage:", Path('/kaggle/working').resolve())


In [ ]:
# Run this cell first to restore the unchanged repository source.
import base64, io, zipfile, os
PROJECT = Path('/kaggle/working/11685-diffusion')
PROJECT.mkdir(parents=True, exist_ok=True)
project_zip = base64.b64decode('...')
with zipfile.ZipFile(io.BytesIO(project_zip)) as z:
    z.extractall(PROJECT)
os.chdir(PROJECT)
print('Project:', PROJECT)
print('Source files:', len(list(PROJECT.rglob('*.py'))))
assert (PROJECT / 'requirements.txt').is_file(), 'Source restoration failed'


In [ ]:
# Install pinned metric dependencies after the source-restore cell succeeds.
assert (PROJECT / 'requirements.txt').is_file(), 'Run the source-restore cell above first'
%pip -q install -r /kaggle/working/11685-diffusion/requirements.txt 'torchmetrics[image]==1.9.0' 'torch-fidelity==0.4.0' gdown

import importlib.metadata as metadata
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
print('torchmetrics:', metadata.version('torchmetrics'))
print('torch-fidelity:', metadata.version('torch-fidelity'))
assert metadata.version('torchmetrics') == '1.9.0'
assert metadata.version('torch-fidelity') == '0.4.0'


In [ ]:
# Download the course data directly into Kaggle's working storage.
import subprocess
archive = PROJECT / 'imagenet100_128x128.tar.gz'
if not archive.exists():
    subprocess.run(['python', '-m', 'gdown', '11Pk4QvNX3fZkpa_ZvdjD_fQFjCBoa1M0', '-O', str(archive)], check=True)
print('Archive bytes:', archive.stat().st_size)
assert archive.stat().st_size > 690_000_000, 'Download is incomplete'


In [ ]:
# Inspect and extract the archive. Reject links and path escapes.
import tarfile
data_root = PROJECT / 'data'
data_root.mkdir(exist_ok=True)
with tarfile.open(archive, 'r:gz') as t:
    members = t.getmembers()
    print('First archive entries:', [m.name for m in members[:8]])
    assert all(not m.issym() and not m.islnk() for m in members), 'Archive contains links'
    t.extractall(data_root, filter='data')
train_dirs = [p for p in data_root.rglob('train') if p.is_dir()]
assert len(train_dirs) == 1, f'Expected one train directory, got {train_dirs}'
TRAIN_DIR = train_dirs[0]
print('Train directory:', TRAIN_DIR)
print('Classes:', len([p for p in TRAIN_DIR.iterdir() if p.is_dir()]))


In [ ]:
# Run retained scaffold tests that still apply after completing the TODOs.
# Three original tests specifically expect unfinished TODO boundaries and are not applicable.
import unittest
obsolete = {
    'test_unfinished_default_config_explains_todos',
    'test_cpu_training_reaches_explicit_exercise_boundary',
    'test_inference_reaches_explicit_exercise_boundary',
}
def walk_tests(group):
    for item in group:
        if isinstance(item, unittest.TestSuite):
            yield from walk_tests(item)
        else:
            yield item

discovered = unittest.defaultTestLoader.discover('tests', pattern='test_*.py')
applicable = [test for test in walk_tests(discovered)
              if test.id().rsplit('.', 1)[-1] not in obsolete]
assert applicable, 'No tests discovered'
print('Skipped obsolete TODO-boundary tests:', sorted(obsolete))
suite = unittest.TestSuite(applicable)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), 'Applicable unit tests failed; inspect the output above'
print(f'Applicable unit tests passed: {result.testsRun}')


## Training protocol

The pilot uses 10 images per class (1,000 total), 64×64 pixels, batch size 2, one epoch, and 50-step sampling. It checks the training and checkpoint path. The planned midterm run uses **all training images** at 64×64, batch size 2, **3 epochs**, and the same model/configuration. `train.py` starts from scratch; it cannot resume the earlier pilot checkpoint through its CLI.

Set `RUN_MODE` below to `full` only after the pilot has been preserved. DDPM and DDIM are sampling methods for a trained checkpoint; they require **zero additional training epochs**. Both evaluation calls below use the same checkpoint and 50 sampling steps for a direct scheduler comparison.


In [ ]:
# Choose 'pilot' (default) or 'full' before starting this notebook.
RUN_MODE = 'pilot'
# To evaluate an already saved pilot checkpoint, set this to its path in this session.
# Leave None to train a new checkpoint using RUN_MODE.
EXISTING_CHECKPOINT = None
RESOLUTION = 64
IMAGES_PER_CLASS = 10
BATCH_SIZE = 2
EPOCHS = 1 if RUN_MODE == 'pilot' else 3
INFERENCE_STEPS = 50
EVAL_IMAGES = 5000
EVAL_BATCH_SIZE = 4
NUM_WORKERS = 2
assert RUN_MODE in ('pilot', 'full')
assert RESOLUTION == 64, 'The validation preprocessing below is documented for 64×64'
assert EVAL_IMAGES == 5000, 'The unchanged inference.py generates 5,000 images'
print(dict(run_mode=RUN_MODE, resolution=RESOLUTION, epochs=EPOCHS,
           batch_size=BATCH_SIZE, inference_steps=INFERENCE_STEPS))


In [ ]:
# Create a balanced ImageFolder pilot; the full run uses TRAIN_DIR directly.
import shutil
pilot_train = PROJECT / 'pilot_data' / 'train'
classes = sorted(p for p in TRAIN_DIR.iterdir() if p.is_dir())
assert len(classes) == 100, f'Expected 100 class folders, got {len(classes)}'
if RUN_MODE == 'pilot':
    if pilot_train.exists():
        shutil.rmtree(pilot_train)
    count = 0
    for cls in classes:
        target = pilot_train / cls.name
        target.mkdir(parents=True)
        files = sorted(p for p in cls.iterdir() if p.is_file())[:IMAGES_PER_CLASS]
        assert len(files) == IMAGES_PER_CLASS, f'Too few images in {cls}'
        for p in files:
            (target / p.name).symlink_to(p)
            count += 1
    TRAINING_DIR = pilot_train
else:
    TRAINING_DIR = TRAIN_DIR
    count = sum(1 for cls in classes for p in cls.iterdir() if p.is_file())
print('Training images:', count, 'across', len(classes), 'classes')
print('Training directory:', TRAINING_DIR)


In [ ]:
# Train with the unchanged entry point. The run name separates pilot and full checkpoints.
import time
env = os.environ.copy()
env['WANDB_MODE'] = 'disabled'
run_name = 'kaggle-baseline-' + RUN_MODE
command = [
    'python', 'train.py', '--config', 'configs/ddpm.yaml',
    '--device', 'cuda', '--data_dir', str(TRAINING_DIR),
    '--image_size', str(RESOLUTION), '--unet_in_size', str(RESOLUTION),
    '--batch_size', str(BATCH_SIZE), '--num_epochs', str(EPOCHS),
    '--num_workers', str(NUM_WORKERS),
    '--num_inference_steps', str(INFERENCE_STEPS),
    '--output_dir', '/kaggle/working/experiments',
    '--run_name', run_name,
]
print(' '.join(command), flush=True)
if EXISTING_CHECKPOINT is None:
    started = time.time()
    subprocess.run(command, cwd=PROJECT, env=env, check=True)
    print('Training wall time (minutes):', round((time.time() - started)/60, 2))
else:
    print('Using existing checkpoint; training skipped:', EXISTING_CHECKPOINT)


In [ ]:
# Select the last epoch from this run only, never a checkpoint from another experiment.
if EXISTING_CHECKPOINT is not None:
    CKPT = Path(EXISTING_CHECKPOINT)
else:
    checkpoints = sorted(
        Path('/kaggle/working/experiments').glob(f'*-{run_name}/checkpoints/checkpoint_epoch_*.pth'),
        key=lambda p: int(p.stem.rsplit('_', 1)[-1]),
    )
    assert checkpoints, 'No checkpoint was saved; inspect the training log above'
    CKPT = checkpoints[-1]
assert CKPT.is_file(), f'Checkpoint not found: {CKPT}'
assert CKPT.stat().st_size > 0
print('Checkpoint:', CKPT, 'bytes:', CKPT.stat().st_size)
print('Download this file or save the Kaggle version before ending the session.')


## Evaluation reference set

The unchanged `inference.py` expects a `validation` folder beside the path passed as `--data_dir` and does **not** resize reference images. To compare 64×64 generated images with 64×64 real images, this cell resizes the course validation split to 64×64 with bilinear interpolation and saves PNGs in a separate folder. The original validation images remain intact. FID is sensitive to this preprocessing, so report it with every score.


In [ ]:
# Prepare the complete, class-preserving 64×64 validation reference set.
from PIL import Image
original_validation = TRAIN_DIR.parent / 'validation'
assert original_validation.is_dir(), f'Missing validation split: {original_validation}'
eval_root = PROJECT / 'eval_data'
eval_train = eval_root / 'train'
eval_validation = eval_root / 'validation'
eval_train.mkdir(parents=True, exist_ok=True)  # inference.py locates ../validation from here
validation_classes = sorted(p for p in original_validation.iterdir() if p.is_dir())
assert len(validation_classes) == 100
source_images = [p for cls in validation_classes for p in cls.iterdir() if p.is_file()]
assert source_images, 'Validation split is empty'
for cls in validation_classes:
    target_dir = eval_validation / cls.name
    target_dir.mkdir(parents=True, exist_ok=True)
    for src in sorted(p for p in cls.iterdir() if p.is_file()):
        dst = target_dir / (src.stem + '.png')
        if not dst.exists():
            with Image.open(src) as image:
                image.convert('RGB').resize((RESOLUTION, RESOLUTION), Image.Resampling.BILINEAR).save(dst)
reference_count = sum(1 for p in eval_validation.rglob('*.png'))
assert reference_count == len(source_images), (reference_count, len(source_images))
print('64×64 validation images:', reference_count)


## DDPM and DDIM: 5,000 generated images each

The two calls below use the same checkpoint, seed, model, 64×64 resolution, and **50 denoising steps**. Only the sampler changes. `inference.py` computes FID against the resized validation set and IS on generated images. Metric model weights download on first use, so Internet must be enabled. Logs and parsed metrics are saved in `/kaggle/working/evaluation`.


In [ ]:
import json
import re
from datetime import datetime, timezone

EVAL_DIR = Path('/kaggle/working/evaluation')
EVAL_DIR.mkdir(parents=True, exist_ok=True)

def evaluate(sampler):
    cmd = [
        'python', 'inference.py', '--config', 'configs/ddpm.yaml',
        '--device', 'cuda', '--data_dir', str(eval_train),
        '--ckpt', str(CKPT), '--image_size', str(RESOLUTION),
        '--unet_in_size', str(RESOLUTION), '--batch_size', str(EVAL_BATCH_SIZE),
        '--num_workers', str(NUM_WORKERS), '--num_inference_steps', str(INFERENCE_STEPS),
        '--use_ddim', 'true' if sampler == 'DDIM' else 'false',
    ]
    print(' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=PROJECT, env=env, text=True,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log = EVAL_DIR / f'{RUN_MODE}_{sampler.lower()}_50steps.log'
    log.write_text(completed.stdout)
    print(completed.stdout[-5000:])
    if completed.returncode != 0:
        raise RuntimeError(f'{sampler} evaluation failed; full log: {log}')
    def metric(pattern):
        matches = re.findall(pattern, completed.stdout)
        assert matches, f'Missing metric {pattern}; inspect {log}'
        return float(matches[-1])
    result = {
        'run_mode': RUN_MODE, 'sampler': sampler, 'checkpoint': str(CKPT),
        'train_epochs': EPOCHS, 'train_images': count,
        'resolution': RESOLUTION, 'sampling_steps': INFERENCE_STEPS,
        'generated_images': int(metric(r'Generated images:\s*(\d+)')),
        'reference_images': reference_count,
        'reference_preprocessing': 'PIL RGB bilinear resize to 64x64, PNG',
        'fid': metric(r'FID:\s*([0-9.eE+-]+)'),
        'is_mean': metric(r'IS:\s*([0-9.eE+-]+)\s*\+/-'),
        'is_std': metric(r'IS:\s*[0-9.eE+-]+\s*\+/-\s*([0-9.eE+-]+)'),
        'sampling_seconds': metric(r'Sampling time:\s*([0-9.eE+-]+) seconds'),
        'completed_utc': datetime.now(timezone.utc).isoformat(),
    }
    assert result['generated_images'] == EVAL_IMAGES
    out = EVAL_DIR / f'{RUN_MODE}_{sampler.lower()}_50steps.json'
    out.write_text(json.dumps(result, indent=2))
    print('Saved metrics:', out)
    return result


In [ ]:
# Evaluate the trained checkpoint with ancestral DDPM sampling.
ddpm_result = evaluate('DDPM')


In [ ]:
# Evaluate the *same* checkpoint with DDIM sampling.
ddim_result = evaluate('DDIM')


In [ ]:
# Export a compact report table for the midterm write-up.
import csv
rows = [ddpm_result, ddim_result]
columns = ['run_mode', 'sampler', 'train_epochs', 'train_images', 'resolution',
           'sampling_steps', 'generated_images', 'reference_images',
           'fid', 'is_mean', 'is_std', 'sampling_seconds', 'checkpoint',
           'reference_preprocessing', 'completed_utc']
report = EVAL_DIR / f'{RUN_MODE}_ddpm_ddim_metrics.csv'
with report.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(rows)
for row in rows:
    print(f"{row['sampler']}: FID={row['fid']:.4f}, IS={row['is_mean']:.4f} "
          f"+/- {row['is_std']:.4f}, sampling={row['sampling_seconds']:.1f}s")
print('CSV:', report)
print('Download the checkpoint, CSV, JSON, and logs or save a Kaggle version.')


## Reporting notes

- State whether results are from the **1,000-image pilot** or the **full ImageNet-100 training split**. Do not label pilot scores as full-data results.
- Report epochs, optimizer steps, batch size, 64×64 resolution, GPU, seed, 1,000 diffusion training timesteps, 50 inference steps, 5,000 generated images per sampler, and the validation resizing described above.
- Lower FID and higher IS are generally better, but a one-epoch pilot chiefly checks that the pipeline works. The two metrics are meaningful only with the associated checkpoint and exact preprocessing recorded.
- Keep the original source and TODO comments with the code submission; this notebook supplies the reproducible execution and metric protocol.
